In [1]:
import sys
import os
# Add the root directory to sys.path to allow importing ssonn
sys.path.append(os.path.abspath("../../.."))


In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    torch.set_deterministic_debug_mode(True)

from ssonn.model.utils import *
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.metrics.train_metrics import *
from ssonn.train.train import *


In [3]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=27 * 18, output_size=6):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc0(x)


In [4]:
USE_MOMENTUM = True # Set to True for SGD + Momentum, False for SGD
CHECKPOINT_PATH = "checkpoint.pth"
RESUME = False # Set to True to resume training
ADDITIONAL_EPOCHS = 0 # Number of epochs to add if resuming

hyperparams = {
    "num_epochs": 256,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.1},
    "pruning_thresholds": {"fc0": 1},
    "plateau_threshold": 0.002,
    "min_epochs_between_expansions": 16,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 1000,
    "weight_decay": 5e-3,
    "diff_threshold": 0.05,
    "adam_betas": (0.9, 0.999),
    "momentum": 0.9
}


In [6]:
train_x = np.load("../data/Gutenberg/train_x.npy")
train_y = np.load("../data/Gutenberg/train_y.npy")
valid_x = np.load("../data/Gutenberg/valid_x.npy")
valid_y = np.load("../data/Gutenberg/valid_y.npy")
test_x = np.load('../data/Gutenberg/test_x.npy')
test_y = np.load('../data/Gutenberg/test_y.npy')

train_tensor_x = torch.tensor(train_x).float().view(-1, 27 * 18)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 27 * 18)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 27 * 18)
test_tensor_y = torch.tensor(test_y).long()

train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)

train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True, generator=g)
val_loader = DataLoader(valid_dataset, batch_size=hyperparams['batch_size'], shuffle=False, generator=g)
test_loader = DataLoader(test_dataset, batch_size=hyperparams['batch_size'], shuffle=False, generator=g)


In [7]:
import os
from dotenv import load_dotenv

load_dotenv()
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="gutenberg-sgd-vs-adam",
)
experiment.log_parameters(hyperparams)
experiment.set_name("SGD_Momentum" if USE_MOMENTUM else "SGD_No_Momentum")


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gcof/gutenberg-sgd-vs-adam/bcec35b3bc6741c2b1a441d32dff9eee



In [ ]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

criterion = nn.CrossEntropyLoss()
# Using SGD with Momentum instead of ADAM
if USE_MOMENTUM:
    optimizer = torch.optim.SGD(sparse_model.parameters(), lr=hyperparams['learning_rate'], momentum=hyperparams['momentum'], weight_decay=hyperparams['weight_decay'])
else:
    optimizer = torch.optim.SGD(sparse_model.parameters(), lr=hyperparams['learning_rate'], momentum=0, weight_decay=hyperparams['weight_decay'])

start_epoch = 0
resume_state = None

if RESUME and os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH)
    sparse_model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    resume_state = checkpoint['resume_state']
    print(f"Resuming from epoch {start_epoch}")

    if ADDITIONAL_EPOCHS > 0:
        hyperparams['num_epochs'] = start_epoch + ADDITIONAL_EPOCHS
        print(f"Training until epoch {hyperparams['num_epochs']}")
else:
    print("Starting fresh training")

# Note: We pass train_loader as the second argument (val_loader in function signature)
# because we want to calculate edge importance based on training data gradients.
# The third argument (test_loader in function signature) is used for validation logging.
final_state = train_sparse_recursive(
    sparse_model,
    train_loader,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    hyperparams,
    device,
    experiment,
    start_epoch=start_epoch,
    resume_state=resume_state
)

# Save checkpoint
torch.save({
    'epoch': hyperparams['num_epochs'],
    'model_state_dict': sparse_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'resume_state': final_state
}, CHECKPOINT_PATH)


Starting fresh training


100%|██████████| 176/176 [00:00<00:00, 335.02it/s]


Epoch 1/256, Train Loss: 1.7947, Val Loss: 1.7922, Val Accuracy: 0.1777


100%|██████████| 176/176 [00:00<00:00, 368.88it/s]


Epoch 2/256, Train Loss: 1.7894, Val Loss: 1.7879, Val Accuracy: 0.1885


100%|██████████| 176/176 [00:00<00:00, 265.36it/s]


... (434 outputs left out)

Epoch 219/256, Train Loss: 1.6684, Val Loss: 1.6925, Val Accuracy: 0.2950


100%|██████████| 176/176 [03:47<00:00,  1.29s/it]
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (8251365 > 1048576). Falling back to PNG format.


Epoch 220/256, Train Loss: 1.6684, Val Loss: 1.6923, Val Accuracy: 0.2957


 96%|█████████▌| 169/176 [03:38<00:08,  1.27s/it]

In [ ]:
experiment.end()


In [ ]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)
print(f"Accuracy: {accuracy}, Params: {params}")
